# 02 — Feature Engineering

Transforming raw CRM tables into a model-ready feature matrix for account-level propensity scoring.

**The business question:** Given what we know about an account (firmographics, tech stack, contact composition, deal history), can we predict which accounts are most likely to close?

**Approach:**
1. Start with account-level firmographic and technographic features
2. Roll up contact-level attributes into account-level composition metrics
3. Engineer buying group signals from deal contact roles
4. Analyze correlations and select features for modeling

Everything here feeds into `src/features.py` — this notebook is the walkthrough of *why* each feature exists.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

# Load raw tables
accounts = pd.read_csv("../data/accounts.csv")
contacts = pd.read_csv("../data/contacts.csv")
opportunities = pd.read_csv("../data/opportunities.csv")
contact_opp = pd.read_csv("../data/contact_opportunity.csv")

print(f"Accounts:      {len(accounts):,}")
print(f"Contacts:      {len(contacts):,}")
print(f"Opportunities: {len(opportunities):,}")
print(f"Contact roles: {len(contact_opp):,}")

## 1. Firmographic & Technographic Features

These come directly from the accounts table — the kind of data you'd pull from Salesforce or a data enrichment vendor like ZoomInfo.

**Why these matter:** In B2B SaaS, company characteristics are the first filter. A 50-person startup and a 10,000-person enterprise have completely different buying patterns, deal sizes, and sales cycles. The tech stack tells you whether the account is already in your ecosystem.

In [ ]:
# --- Segment encoding ---
# One-hot encode segment (SMB / Mid-Market / Enterprise)
# Why not ordinal? The relationship isn't strictly linear — Mid-Market has the highest
# win rate, not Enterprise. One-hot lets the model learn each segment's effect independently.

seg_dummies = pd.get_dummies(accounts["segment"], prefix="seg")
print("Segment encoding:")
print(seg_dummies.head(8).to_string())
print(f"\nSegment distribution:\n{seg_dummies.sum().to_string()}")

In [ ]:
# --- Size features (log-transformed) ---
# Employee count and revenue are heavily right-skewed (log-normal).
# Log transform compresses the range so a 50-person company and a 20,000-person company
# aren't treated as 400x different — the marginal effect of size diminishes.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label in zip(axes, ["employee_count", "annual_revenue"], ["Employee Count", "Annual Revenue"]):
    ax.hist(accounts[col], bins=50, alpha=0.5, label="Raw", color="steelblue")
    ax2 = ax.twinx()
    ax2.hist(np.log1p(accounts[col]), bins=50, alpha=0.5, label="log(1+x)", color="coral")
    ax.set_xlabel(label)
    ax.set_ylabel("Raw frequency", color="steelblue")
    ax2.set_ylabel("Log frequency", color="coral")
    ax.set_title(f"{label}: Raw vs. Log Transform")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

print(f"Employee count — raw range: [{accounts['employee_count'].min():,}, {accounts['employee_count'].max():,}]")
print(f"Employee count — log range: [{np.log1p(accounts['employee_count'].min()):.1f}, {np.log1p(accounts['employee_count'].max()):.1f}]")

In [ ]:
# --- Technographic features ---
# The tech stack is a comma-separated string. We extract two kinds of signals:
# 1. Count of "high-signal" tools (complementary tech that suggests product fit)
# 2. Individual flags for the most common tools

HIGH_SIGNAL_TOOLS = {"Jira", "Salesforce", "Slack", "Snowflake", "AWS"}
TOP_TOOLS = ["Salesforce", "Slack", "Jira", "AWS", "Azure", "GCP", "HubSpot",
             "Snowflake", "Databricks", "Tableau"]

tech_stacks = accounts["tech_stack"].apply(lambda ts: set(t.strip() for t in str(ts).split(",")))

tech_df = pd.DataFrame({
    "account_id": accounts["account_id"],
    "tech_stack_count": tech_stacks.apply(len),
    "high_signal_tool_count": tech_stacks.apply(lambda tools: len(tools & HIGH_SIGNAL_TOOLS)),
})

# Individual tool flags
for tool in TOP_TOOLS:
    tech_df[f"has_{tool.lower()}"] = tech_stacks.apply(lambda tools, t=tool: int(t in tools))

print("Tech feature summary:")
print(tech_df.describe().round(2).to_string())
print(f"\nHigh-signal tool distribution:")
print(tech_df["high_signal_tool_count"].value_counts().sort_index().to_string())

In [ ]:
# --- Existing customer features ---
# Upsell/cross-sell to existing customers is usually easier than net-new.
# ARR gives the relationship depth — a $5K customer and a $200K customer are different signals.

existing = accounts[["account_id", "has_existing_product", "arr"]].copy()
existing["has_existing_product"] = existing["has_existing_product"].astype(int)
existing["arr_log"] = np.log1p(existing["arr"])

print(f"Existing customers: {existing['has_existing_product'].sum()} / {len(existing)} ({existing['has_existing_product'].mean():.1%})")
print(f"\nARR for existing customers:")
print(accounts[accounts["has_existing_product"]]["arr"].describe().apply(lambda x: f"${x:,.0f}").to_string())

## 2. Contact Composition Features

This is where it gets interesting. Raw account firmographics tell you *what* a company looks like. Contact composition tells you *who you're actually talking to* at that company — and that's a much stronger signal for whether a deal will close.

We roll up contact-level attributes to the account level:
- **Contact count** — how many people do we know at this account?
- **Seniority distribution** — VP+ count, director+ count, max seniority, "senior density"
- **Function diversity** — are we talking to just one department, or across the org?
- **Technical + Business mix** — deals close faster when both the builders and the buyers are involved

In [ ]:
TECHNICAL_FUNCTIONS = {"Engineering", "IT", "Product"}
BUSINESS_FUNCTIONS = {"Marketing", "Sales", "Finance", "Executive"}
SENIORITY_RANK = {
    "Individual Contributor": 1, "Manager": 2, "Director": 3, "VP": 4, "C-Suite": 5,
}

contact_agg = contacts.groupby("account_id").agg(
    contact_count=("contact_id", "count"),
    vp_plus_count=("seniority", lambda s: (s.isin(["C-Suite", "VP"])).sum()),
    director_plus_count=("seniority", lambda s: (s.isin(["C-Suite", "VP", "Director"])).sum()),
    function_diversity=("job_function", "nunique"),
    has_technical=("job_function", lambda f: int(any(x in TECHNICAL_FUNCTIONS for x in f))),
    has_business=("job_function", lambda f: int(any(x in BUSINESS_FUNCTIONS for x in f))),
    max_seniority=("seniority", lambda s: max(SENIORITY_RANK.get(x, 0) for x in s)),
).reset_index()

contact_agg["has_technical_and_business"] = (
    (contact_agg["has_technical"] == 1) & (contact_agg["has_business"] == 1)
).astype(int)
contact_agg["senior_density"] = contact_agg["vp_plus_count"] / contact_agg["contact_count"]

print("Contact composition features:")
print(contact_agg.describe().round(2).to_string())

In [ ]:
# Visualize key contact composition features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(contact_agg["contact_count"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Contacts per Account")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Contact Count Distribution")

axes[1].hist(contact_agg["function_diversity"], bins=range(1, 10), color="steelblue",
             edgecolor="white", align="left")
axes[1].set_xlabel("Distinct Job Functions")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Function Diversity")

axes[2].hist(contact_agg["senior_density"], bins=20, color="steelblue", edgecolor="white")
axes[2].set_xlabel("VP+ / Total Contacts")
axes[2].set_ylabel("Frequency")
axes[2].set_title("Senior Density")

plt.tight_layout()
plt.show()

print(f"Accounts with both technical AND business contacts: "
      f"{contact_agg['has_technical_and_business'].sum()} / {len(contact_agg)} "
      f"({contact_agg['has_technical_and_business'].mean():.1%})")

## 3. Deal Contact / Buying Group Features

The strongest propensity signals come from who's *already engaged* on a deal. This is the buying group layer — going beyond "who's in our database" to "who's actively involved in the sales process."

We look at:
- **Deal contact count** — how many distinct people are on deals at this account?
- **Role diversity** — are there Champions, Decision Makers, Evaluators, and Influencers?
- **Role coverage** — what fraction of the 4 key buying roles are present?
- **Buying group completeness** — a composite score (0-100) combining role coverage, seniority presence, and function diversity on the deal

In [ ]:
# Enrich the bridge table with contact attributes and deal info
deal_contacts = (
    contact_opp
    .merge(contacts[["contact_id", "seniority", "job_function"]], on="contact_id")
    .merge(opportunities[["opportunity_id", "account_id", "is_won"]], on="opportunity_id")
)

KEY_ROLES = ["Champion", "Decision Maker", "Evaluator", "Influencer"]

# Aggregate to account level
acct_deal = deal_contacts.groupby("account_id").agg(
    deal_contact_count=("contact_id", "nunique"),
    deal_role_diversity=("role", "nunique"),
    has_champion=("role", lambda r: int("Champion" in set(r))),
    has_decision_maker=("role", lambda r: int("Decision Maker" in set(r))),
    has_evaluator=("role", lambda r: int("Evaluator" in set(r))),
    has_influencer=("role", lambda r: int("Influencer" in set(r))),
    deal_vp_plus=("seniority", lambda s: int(any(x in ["C-Suite", "VP"] for x in s))),
    deal_function_diversity=("job_function", "nunique"),
).reset_index()

# Role coverage (0-1 scale)
role_coverage = deal_contacts.groupby("account_id")["role"].apply(
    lambda r: len(set(r) & set(KEY_ROLES)) / len(KEY_ROLES)
).reset_index(name="role_coverage")
acct_deal = acct_deal.merge(role_coverage, on="account_id", how="left")

# Buying group completeness score (0-100)
acct_deal["buying_group_completeness"] = (
    acct_deal["role_coverage"] * 25
    + acct_deal["deal_vp_plus"] * 25
    + (acct_deal["deal_function_diversity"] >= 2).astype(int) * 25
    + acct_deal.apply(
        lambda row: 25 if row.get("has_champion", 0) and row.get("has_decision_maker", 0) else 0,
        axis=1
    )
)

print("Buying group features (accounts with deals only):")
print(acct_deal.describe().round(2).to_string())

In [ ]:
# Visualize the buying group completeness distribution and its relationship to deal outcomes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Completeness distribution
axes[0].hist(acct_deal["buying_group_completeness"], bins=15, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Buying Group Completeness Score")
axes[0].set_ylabel("Accounts")
axes[0].set_title("Buying Group Completeness Distribution")

# Completeness vs win rate
closed = opportunities[opportunities["stage"].isin(["Closed Won", "Closed Lost"])]
acct_outcome = closed.groupby("account_id")["is_won"].max().reset_index()
merged = acct_deal.merge(acct_outcome, on="account_id", how="inner")

merged["completeness_tier"] = pd.cut(
    merged["buying_group_completeness"],
    bins=[-1, 25, 50, 75, 100],
    labels=["Low\n(0-25)", "Medium\n(25-50)", "High\n(50-75)", "Complete\n(75-100)"],
)
tier_stats = merged.groupby("completeness_tier", observed=False)["is_won"].agg(["mean", "count"])

bars = axes[1].bar(range(len(tier_stats)), tier_stats["mean"], color="steelblue", edgecolor="white")
axes[1].set_xticks(range(len(tier_stats)))
axes[1].set_xticklabels(tier_stats.index)
axes[1].set_ylabel("Win Rate")
axes[1].set_title("Win Rate by Buying Group Completeness")
axes[1].set_ylim(0, 0.6)

for i, (bar, (_, row)) in enumerate(zip(bars, tier_stats.iterrows())):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{row["mean"]:.0%}\n(n={row["count"]:.0f})', ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 4. Assembling the Full Feature Matrix

Now we combine all three feature layers into a single account-level dataset, using `src/features.py` which packages everything above into reusable functions.

This gives us one row per account with closed deals, and a binary target: did any deal at this account close-won?

In [ ]:
from src.features import get_modeling_dataset

X, y = get_modeling_dataset(accounts, contacts, opportunities, contact_opp)

print(f"Modeling dataset: {X.shape[0]} accounts, {X.shape[1]} features")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Win rate (positive class): {y.mean():.1%}")
print(f"\nFeatures ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Quick sanity checks
print("=== Sanity Checks ===")
print(f"Any NaN values: {X.isna().any().any()}")
print(f"Any infinite values: {np.isinf(X.select_dtypes(include=[np.number])).any().any()}")
print(f"Target leak check — 'account_id' in features: {'account_id' in X.columns}")
print(f"Target leak check — 'target' in features: {'target' in X.columns}")
print(f"\nFeature dtypes:")
print(X.dtypes.value_counts().to_string())

## 5. Correlation Analysis

Two things to look for:
1. **Feature-target correlations** — which features actually predict wins? This validates our hypotheses from the EDA notebook.
2. **Feature-feature correlations** — highly correlated features add redundancy without improving the model. Logistic regression handles this okay with regularization, but it's worth knowing about.

In [ ]:
# Feature-target correlation (point-biserial since target is binary)
target_corr = X.corrwith(y).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in target_corr]
ax.barh(range(len(target_corr)), target_corr.values, color=colors)
ax.set_yticks(range(len(target_corr)))
ax.set_yticklabels(target_corr.index, fontsize=8)
ax.set_xlabel("Correlation with Target (Won)")
ax.set_title("Feature-Target Correlations")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 positive correlations with target:")
for feat, corr in target_corr.head(10).items():
    print(f"  {feat:40s} {corr:+.3f}")
print("\nTop 5 negative correlations:")
for feat, corr in target_corr.tail(5).items():
    print(f"  {feat:40s} {corr:+.3f}")

In [ ]:
# Feature-feature correlation heatmap
# Focus on the most important features to keep it readable
top_features = target_corr.abs().sort_values(ascending=False).head(15).index.tolist()
corr_matrix = X[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax, square=True,
    xticklabels=True, yticklabels=True,
    annot_kws={"fontsize": 8},
)
ax.set_title("Feature-Feature Correlations (Top 15 by Target Correlation)")
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("Highly correlated feature pairs (|r| > 0.7):")
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f"  {f1:35s} <-> {f2:35s}  r={r:+.2f}")
else:
    print("No feature pairs with |r| > 0.7 among top features.")

## 6. Feature Selection

We have a lot of features relative to our sample size. Some are redundant (industry and segment dummies, individual tool flags that overlap with `high_signal_tool_count`). Rather than manually pruning, we'll rely on two mechanisms:

1. **L2 regularization** in logistic regression — shrinks coefficients of redundant/weak features toward zero
2. **Post-hoc inspection** — after training in notebook 03, we'll look at which features the model actually uses

For now, let's categorize what we have and do a quick variance check to catch any zero-variance columns.

In [ ]:
# Categorize features
categories = {
    "Segment": [c for c in X.columns if c.startswith("seg_")],
    "Industry": [c for c in X.columns if c.startswith("ind_")],
    "Region": [c for c in X.columns if c.startswith("region_")],
    "Size": ["employee_count_log", "annual_revenue_log"],
    "Existing Customer": ["has_existing_product", "arr_log"],
    "Tech Stack": ["tech_stack_count", "high_signal_tool_count"] + [c for c in X.columns if c.startswith("has_") and c not in ["has_existing_product", "has_technical", "has_business", "has_technical_and_business", "has_champion", "has_decision_maker", "has_evaluator", "has_influencer"]],
    "Contact Composition": ["contact_count", "vp_plus_count", "director_plus_count", "function_diversity",
                           "has_technical", "has_business", "has_technical_and_business",
                           "max_seniority", "senior_density"],
    "Buying Group": ["deal_contact_count", "deal_role_diversity", "has_champion", "has_decision_maker",
                     "has_evaluator", "has_influencer", "deal_vp_plus",
                     "deal_function_diversity", "role_coverage", "buying_group_completeness"],
}

print(f"Feature categories ({sum(len(v) for v in categories.values())} total):\n")
for cat, feats in categories.items():
    present = [f for f in feats if f in X.columns]
    missing = [f for f in feats if f not in X.columns]
    print(f"  {cat} ({len(present)} features): {', '.join(present)}")
    if missing:
        print(f"    ⚠ Not in X: {', '.join(missing)}")

In [ ]:
# Variance check — zero-variance features carry no information
variances = X.var().sort_values()
low_var = variances[variances < 0.01]

if len(low_var) > 0:
    print(f"Low variance features (var < 0.01): {len(low_var)}")
    for feat, var in low_var.items():
        print(f"  {feat:40s} var={var:.4f}  unique={X[feat].nunique()}")
    print("\nThese won't hurt a regularized model, but they won't help much either.")
else:
    print("No zero/near-zero variance features — all features carry some signal.")

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Samples-to-features ratio: {X.shape[0] / X.shape[1]:.1f}x")

## 7. Train/Test Split Preview

Before heading to the modeling notebook, let's verify the train/test split preserves the target distribution. We'll use 80/20 stratified split — the same one the modeling notebook will use.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} accounts ({y_train.mean():.1%} win rate)")
print(f"Test:  {len(X_test)} accounts ({y_test.mean():.1%} win rate)")
print(f"Overall: {len(X)} accounts ({y.mean():.1%} win rate)")
print(f"\n✓ Stratified split preserves class balance")

## Summary

We've built a feature matrix with three layers of signal, from weakest to strongest:

| Layer | What it captures | Example features |
|-------|-----------------|-----------------|
| **Firmographic/Technographic** | What kind of company is this? | segment, employee count, tech stack, existing customer |
| **Contact Composition** | Who do we know there? | VP+ count, function diversity, senior density |
| **Buying Group Signals** | Who's actively engaged on a deal? | role coverage, completeness score, Champion + Decision Maker presence |

**Key observations:**
- Buying group features (deal contact composition, role coverage) show the strongest target correlations — validating the hypothesis from notebook 01
- VP+ presence and tech/business function mix are strong contact-level signals
- Firmographic features (segment, industry) provide useful context but are weaker individually
- Regularization will handle feature redundancy (e.g., `vp_plus_count` and `senior_density` are related)

**Next:** Notebook 03 — train logistic regression and random forest models, evaluate with business metrics (lift charts, precision@K, revenue capture), and interpret the coefficients.